# SparkClient: Interactive Spark Connect (KEP-107 Three-Level API Pattern)

This notebook demonstrates the three levels of the `SparkClient` interactive session API as specified in KEP-107:

1. **Level 1 (Minimal)**: Use defaults for zero-configuration interactive sessions.
2. **Level 2 (Simple)**: Configure basic executor counts and resource requests directly.
3. **Level 3 (Advanced)**: Use granular `Driver` and `Executor` objects for custom pod controls.
4. **Connecting to Existing Sessions**: Attach to an external or running Spark Connect server via base URL.

## 1. Environment and Imports

Import necessary types and configure backend settings.

In [ ]:
import os
import uuid

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import Driver, Executor, Name, SparkClient

namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)
client = SparkClient(backend_config=backend_config)

print(f"SparkClient initialized for namespace: {namespace}")

## 2. Level 1: Minimal Usage

Spin up a Spark Connect session using defaults for executor counts and resource limits.

In [ ]:
session_name = f"spark-connect-minimal-{uuid.uuid4().hex[:8]}"

spark = client.connect(
    spark_conf={"spark.serializer": "org.apache.spark.serializer.KryoSerializer"},
    options=[Name(session_name)],
)

df = spark.range(10)
print(f"Generated range with {df.count()} rows on session: {session_name}")
df.show()

spark.stop()
client.delete_session(session_name)
print("Level 1 session stopped and cleaned up.")

## 3. Level 2: Simple Parameters

Customize compute sizing by passing `num_executors`, `resources_per_executor`, and custom Spark settings.

In [ ]:
session_name = f"my-simple-session-{uuid.uuid4().hex[:8]}"

spark = client.connect(
    num_executors=1,
    resources_per_executor={"cpu": "1", "memory": "512m"},
    spark_conf={
        "spark.sql.adaptive.enabled": "true",
        "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    },
    options=[Name(session_name)],
)

df = spark.range(100)
print(f"Generated range with {df.count()} rows across 1 executor")
df.show(10)

spark.stop()
client.delete_session(session_name)
print("Level 2 session stopped and cleaned up.")

## 4. Level 3: Advanced Driver and Executor Configuration

Leverage `Driver` and `Executor` objects for explicit resource requests, custom service accounts, or GPU targets.

In [ ]:
session_name = f"advanced-session-{uuid.uuid4().hex[:8]}"

spark = client.connect(
    driver=Driver(
        resources={"cpu": "1", "memory": "512m"},
    ),
    executor=Executor(
        num_instances=1,
        resources_per_executor={"cpu": "1", "memory": "512m"},
    ),
    spark_conf={
        "spark.app.name": "advanced-spark-app",
        "spark.sql.adaptive.enabled": "true",
        "spark.sql.adaptive.coalescePartitions.enabled": "true",
        "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    },
    options=[Name(session_name)],
)

df = spark.range(1000)
print(f"Generated range with {df.count()} rows across 1 executor")
df.show(10)

spark.stop()
client.delete_session(session_name)
print("Level 3 session stopped and cleaned up.")

## 5. Connect to an Existing Spark Connect Server

Connect directly to a pre-existing or remote Spark Connect endpoint via `base_url`.

In [ ]:
# Example connection pattern to an existing remote endpoint:
# spark = client.connect(base_url="sc://spark-server:15002")
print("Existing server connection pattern: client.connect(base_url='sc://spark-server:15002')")